In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score,auc
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score

In [5]:
df = pd.read_csv("ICMP ATTACK DATASET.csv")
df.shape

(85063, 84)

In [6]:
df.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
       'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
       'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std',
       'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean',
       'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot',
       'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
       'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',
       'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags',
       'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s',
       'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean',
       'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt',
       'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt',
       'CWE Flag Count', 'ECE 

In [8]:
df.Label.value_counts()

Label
DDOS      65602
NORMAL    19460
Name: count, dtype: int64

Almost 3:1 DDOS to normal ratio. This is fine as later we can add normal traffic which is easier to generate.

In [ ]:
df = df.drop(columns = ['Timestamp','Flow ID','Src IP','Dst IP'])

In [10]:
df = df.dropna()

In [11]:
df.Label.value_counts()

Label
DDOS      65602
NORMAL    19443
Name: count, dtype: int64

In [15]:
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
df.head()

,Src Port,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,39168.0,443.0,6.0,10301270.0,5.0,7.0,517.0,4409.0,517.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,39170.0,443.0,6.0,13284978.0,5.0,7.0,517.0,4411.0,517.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,39170.0,443.0,6.0,169055.0,1.0,2.0,74.0,0.0,74.0,74.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,44264.0,53.0,17.0,63565.0,0.0,2.0,0.0,137.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,36943.0,443.0,17.0,1611997.0,13.0,19.0,5379.0,8967.0,1250.0,33.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [16]:
cormat = df.corr()
cormat = cormat.dropna(how = 'all', axis=1)
cormat = cormat.dropna(how = 'all', axis=0)
cormat.style.background_gradient(cmap='coolwarm')

,Src Port,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Bwd PSH Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
Src Port,1.000000,-0.287051,0.196006,-0.017942,-0.009085,-0.008021,0.035079,0.029646,0.126314,0.080937,0.120697,0.121977,0.163764,0.108473,0.165792,0.169571,-0.012095,-0.124591,-0.002520,0.034602,0.065600,-0.001039,-0.030104,0.000629,0.052215,0.057729,-0.018214,0.017483,0.029354,0.104577,0.118932,0.017558,0.034522,0.027509,0.030720,-0.060698,-0.172607,0.123348,0.163033,0.179361,0.172875,0.116594,-0.189618,0.146303,0.008387,0.034522,-0.075495,0.013124,0.188970,0.120697,0.165792,-0.009085,0.035079,-0.008021,0.029644,0.025263,0.041809,0.008231,0.019477,0.017241,0.000870,0.087772,0.006610,0.067376,0.090953,0.155813
Dst Port,-0.287051,1.000000,-0.021908,-0.253924,-0.001390,-0.001612,-0.005693,-0.006106,-0.018170,0.023070,-0.002712,-0.017143,-0.028302,0.012102,-0.024822,-0.029750,0.041808,0.406183,-0.067406,-0.087911,-0.071044,0.012088,-0.248014,-0.123111,-0.015948,-0.074767,-0.131330,-0.271806,-0.162126,-0.061730,-0.084392,-0.004165,0.037863,-0.034472,-0.020093,0.198822,0.561843,0.003455,-0.027151,-0.022397,-0.028592,-0.016602,0.733042,-0.029410,-0.002024,0.037863,0.587322,-0.296742,-0.021895,-0.002712,-0.024822,-0.001390,-0.005693,-0.001612,-0.006105,0.057859,-0.004709,-0.006816,-0.008026,-0.008384,-0.004153,-0.013769,-0.010525,-0.014818,-0.012660,0.338775
Protocol,0.196006,-0.021908,1.000000,-0.364913,-0.010803,-0.006040,0.080010,0.098773,0.120600,0.461527,0.225255,0.081115,0.125800,0.658091,0.302554,0.141278,0.005229,-0.023402,-0.076498,-0.104959,-0.048894,0.044743,-0.354356,-0.205364,0.027488,-0.063980,-0.241108,-0.442450,-0.289102,-0.103567,-0.123027,0.048119,-0.024999,-0.021389,0.062209,-0.010999,-0.032796,0.676461,0.104864,0.394891,0.159265,0.047824,-0.065646,-0.075441,-0.005182,-0.024999,-0.094820,-0.390207,0.451885,0.225255,0.302554,-0.010803,0.080010,-0.006040,0.098760,-0.027325,0.171366,0.073632,0.064046,0.092020,0.051356,0.022193,0.054380,0.043234,0.010669,0.603098
Flow Duration,-0.017942,-0.253924,-0.364913,1.000000,0.008848,0.013025,0.109754,0.112516,0.168605,-0.107734,0.094565,0.160163,0.126864,-0.207400,0.028950,0.067036,-0.015187,-0.117197,0.444934,0.649564,0.732467,0.075081,0.937400,0.606597,0.588899,0.751453,0.373798,0.886405,0.586591,0.590439,0.678035,-0.003940,0.230272,0.150926,0.147731,-0.057945,-0.161571,-0.232561,0.167964,0.034189,0.119754,0.127561,-0.295802,0.115461,-0.024686,0.230272,-0.070374,0.404679,-0.000258,0.094565,0.028950,0.008848,0.109754,0.013025,0.112503,0.037960,0.126423,0.219962,0.300801,0.295684,0.113308,0.649527,0.482885,0.685093,0.564790,-0.225520
Tot Fwd Pkts,-0.009085,-0.001390,-0.010803,0.008848,1.000000,0.999054,0.003900,0.010275,0.001536,-0.000668,-0.000339,0.000050,0.002120,-0.001339,0.003894,0.000647,-0.000017,0.001870,-0.003437,-0.003434,-0.001393,-0.000579,0.011020,-0.004852,-0.000707,-0.001216,-0.004891,0.009498,-0.005764,-0.002326,-0.001202,-0.000208,-0.000210,0.011127,0.010859,0.002137,0.001446,-0.001375,0.001911,0.003727,0.001776,0.002336,-0.001586,0.000357,-0.000121,-0.000210,-0.002060,0.004837,0.003341,-0.000339,0.003894,1.000000,0.003900,0.999054,0.010276,-0.000058,0.007264,0.003588,0.003666,0.004527,0.002360,0.000300

In [22]:
cormat.shape

(66, 66)

In [17]:
def getCorrelationFeature(corrdata,threshold):
    feature=[]
    value=[]
    for i,index in enumerate(corrdata.index):
        if abs(corrdata[index])>threshold:
            feature.append(index)
            value.append(corrdata[index])
    df=pd.DataFrame(data=value,index=feature,columns=['corr values'])
    return df

In [26]:
threshold=0.50
corr_value=getCorrelationFeature(cormat['Label'],threshold)
corr_value

,corr values
Protocol,0.603098
Pkt Len Mean,0.545784
Pkt Len Std,0.501338
ACK Flag Cnt,0.526085
Pkt Size Avg,0.577708
Label,1.000000


In [27]:
len(corr_value)

6

Six features are too less let's adjust the threhold to 0.4 to so that we get atleast 15-20 features

In [28]:
threshold=0.40
corr_value=getCorrelationFeature(cormat['Label'],threshold)
corr_value

,corr values
Protocol,0.603098
Bwd Pkt Len Max,0.475832
Bwd Pkt Len Min,0.405274
Bwd Pkt Len Mean,0.496963
Bwd Pkt Len Std,0.488018
Pkt Len Min,0.414263
Pkt Len Max,0.469885
Pkt Len Mean,0.545784
Pkt Len Std,0.501338
SYN Flag Cnt,0.419342


In [29]:
len(corr_value)

16

In [30]:
correlated_data=df[corr_value.index]
correlated_data.head()

,Protocol,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,SYN Flag Cnt,ACK Flag Cnt,Pkt Size Avg,Bwd Seg Size Avg,Idle Mean,Idle Max,Label
0,6.0,2860.0,0.0,629.857143,1121.166123,0.0,2860.0,378.923077,852.439388,1.0,0.0,410.500000,629.857143,0.0,0.0,0
1,6.0,1521.0,0.0,630.142857,786.382101,0.0,1521.0,379.076923,638.301060,1.0,0.0,410.666667,630.142857,0.0,0.0,0
2,6.0,0.0,0.0,0.000000,0.000000,0.0,74.0,18.500000,37.000000,0.0,1.0,24.666667,0.000000,0.0,0.0,0
3,17.0,93.0,44.0,68.500000,34.648232,44.0,93.0,60.333333,28.290163,0.0,0.0,90.500000,68.500000,0.0,0.0,0
4,17.0,1250.0,25.0,471.947368,526.736543,25.0,1250.0,472.606061,535.845298,0.0,0.0,487.375000,471.947368,0.0,0.0,0


In [31]:
x=correlated_data.drop(labels=['Label'],axis=1)
y=correlated_data['Label']
x.head()

,Protocol,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,SYN Flag Cnt,ACK Flag Cnt,Pkt Size Avg,Bwd Seg Size Avg,Idle Mean,Idle Max
0,6.0,2860.0,0.0,629.857143,1121.166123,0.0,2860.0,378.923077,852.439388,1.0,0.0,410.500000,629.857143,0.0,0.0
1,6.0,1521.0,0.0,630.142857,786.382101,0.0,1521.0,379.076923,638.301060,1.0,0.0,410.666667,630.142857,0.0,0.0
2,6.0,0.0,0.0,0.000000,0.000000,0.0,74.0,18.500000,37.000000,0.0,1.0,24.666667,0.000000,0.0,0.0
3,17.0,93.0,44.0,68.500000,34.648232,44.0,93.0,60.333333,28.290163,0.0,0.0,90.500000,68.500000,0.0,0.0
4,17.0,1250.0,25.0,471.947368,526.736543,25.0,1250.0,472.606061,535.845298,0.0,0.0,487.375000,471.947368,0.0,0.0


In [32]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=30)

In [33]:
print(x_train.shape,x_test.shape)
print(y_train.shape,y_test.shape)

(68036, 15) (17009, 15)
(68036,) (17009,)


In [34]:
scaler = StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.fit_transform(x_test)

print(x_test.shape)
print(x_train.shape)

(17009, 15)
(68036, 15)


In [52]:
y_train.value_counts()

Label
0    52459
1    15577
Name: count, dtype: int64

## Training Isolation Forest and OCSVM on normal data

In [53]:
X_train_normal = x_train[y_train == 1]

iso = IsolationForest(contamination=0.2, random_state=42)
iso.fit(X_train_normal)

y_pred_iso = iso.predict(x_test)

y_pred_iso = [0 if x == -1 else 1 for x in y_pred_iso]

In [54]:
ocsvm = OneClassSVM(kernel='rbf', nu=0.23, gamma='scale')

ocsvm.fit(X_train_normal)

y_pred_ocsvm = ocsvm.predict(x_test)

# Convert labels
y_pred_ocsvm = [0 if x == -1 else 1 for x in y_pred_ocsvm]

In [41]:
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=3.3  # handle imbalance (~77:23)
)

xgb.fit(x_train, y_train)

y_pred_xgb = xgb.predict(x_test)

In [42]:
def evaluate_model(name, y_true, y_pred):
    print(f"\n{name}")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))

In [55]:
evaluate_model("Isolation Forest", y_test, y_pred_iso)
evaluate_model("One-Class SVM", y_test, y_pred_ocsvm)
evaluate_model("XGBoost", y_test, y_pred_xgb)


Isolation Forest
[[    3 13140]
 [  831  3035]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     13143
           1       0.19      0.79      0.30      3866

    accuracy                           0.18     17009
   macro avg       0.10      0.39      0.15     17009
weighted avg       0.05      0.18      0.07     17009


One-Class SVM
[[   13 13130]
 [ 1834  2032]]
              precision    recall  f1-score   support

           0       0.01      0.00      0.00     13143
           1       0.13      0.53      0.21      3866

    accuracy                           0.12     17009
   macro avg       0.07      0.26      0.11     17009
weighted avg       0.04      0.12      0.05     17009


XGBoost
[[13127    16]
 [    1  3865]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13143
           1       1.00      1.00      1.00      3866

    accuracy                           1.00    

since the normal training data is pretty low one class SVM and Isolation forest model's struggled. Unsupervised might not be useful on this dataset